# GP runtime test

compares the runtime of different ways to evaluate the geometric product of two multivectors, all on the real `PGA3D` tables.

In [5]:
import numpy as np

from pga_engine.algebra import PGA3D
from pga_engine.products import gp

T = PGA3D.gp_table
N = PGA3D.dim
rng = np.random.default_rng(0)
v1, v2 = rng.normal(size=(2, N))

# the classic 2D cayley table: result index and sign per pair (i, j). the result blade is just the XOR of the bitmasks,
# so it is well defined even when the product is 0 (the sign is 0 then)
cayley2d = np.array([[PGA3D.blade_index[bi ^ bj] for bj in PGA3D.blades] for bi in PGA3D.blades])
signs2d = T.sum(axis=2)

## the methods

1. **einsum** on the scattered 3D table (what `gp` does)
2. **np.add.at**: scatter the signed outer product into the result indices of the 2D table
3. **np.bincount**: the same scatter, but via a weighted histogram
4. **gather**: for fixed i and k there is exactly one j with blade_i blade_j = ±blade_k (j = i XOR k on the bitmasks). so instead of scattering, each result coefficient can gather its terms: `result_k = sum_i sign[k, i] v1_i v2_{j(k, i)}`

In [6]:
def gp_einsum(a, b):
    return np.einsum("i, j, ijk -> k", a, b, T)

def gp_add_at(a, b):
    out = np.zeros(N)
    np.add.at(out, cayley2d, signs2d * np.outer(a, b))
    return out

def gp_bincount(a, b):
    return np.bincount(cayley2d.ravel(), weights=(signs2d * np.outer(a, b)).ravel(), minlength=N)

# gather tables: gather_j[k, i] is the j that lands on k, gather_s[k, i] its sign
gather_j = np.empty((N, N), int)
gather_s = np.empty((N, N))
for i in range(N):
    for j in range(N):
        k = cayley2d[i, j]
        gather_j[k, i] = j
        gather_s[k, i] = signs2d[i, j]

def gp_gather(a, b):
    return (gather_s * a[None, :] * b[gather_j]).sum(axis=-1)

# all of them have to give the same result as the library
for f in (gp_einsum, gp_add_at, gp_bincount, gp_gather):
    assert np.allclose(f(v1, v2), gp(v1, v2)), f.__name__
print("all methods agree")

all methods agree


## timings

In [7]:
%timeit gp_einsum(v1, v2)
%timeit gp_add_at(v1, v2)
%timeit gp_bincount(v1, v2)
%timeit gp_gather(v1, v2)

9.04 μs ± 74.3 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
10.6 μs ± 173 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
6.71 μs ± 49.6 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
6.56 μs ± 246 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


batched: 1000 pairs at once. einsum handles the batch dimension directly through `...`

In [4]:
A, B = rng.normal(size=(2, 1000, N))
%timeit gp(A, B)

5.45 ms ± 46.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## conclusion

for a small dim like 16 the methods really aren't that different. the einsum version is the easiest to use with batching (e.g. in a neural network), so that's what `gp` uses.